## 0. Init libraries en helper functions

In [83]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [84]:
import pandas as pd
import re
import numpy as np

# from arcgis.gis import GIS
# from arcgis.apps import survey123

import sqlite3
from datetime import datetime

In [85]:
import os
import sys

# Get the absolute path of the folder above the notebook
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from src import utils

In [86]:
pd.options.display.max_columns = None

SQLITE_PATH = "../input/LSVIHabitatTypes.sqlite"

### 0.1 Inladen gegevens

Gebruikt nu nog exports via LSVI-package (R). Testen of export uit sqlite db kan gemaakt worden.

In [147]:
df_vereisten = pd.read_excel('../input/LSVI_packageInvoervereisten_uitdb_2026-06-08_aanvullingenLSVI_app.xlsx', sheet_name='LSVI_packageInvoervereisten_uit')
df_vereisten["Habitattype"] = df_vereisten["Habitattype"].astype(str).str.strip()
df_vereisten["BeoordelingID"] = df_vereisten["BeoordelingID"].astype(int)
df_vereisten["TaxongroepId"] = df_vereisten["TaxongroepId"].fillna(-1).astype(int)

# We should only use vereiesten v3
df_vereisten = df_vereisten[df_vereisten['Versie'] == 'Versie 3']
print(df_vereisten.shape)
# Maak unieke ID aan voor vragen adhv voorwaarde id + habitattype
df_vereisten['vraag_id'] = "vrg_" + df_vereisten['VoorwaardeID'].astype(str) + "_" + (df_vereisten['Habitatsubtype'].astype(str).apply(utils.clean_name))
# Make sure vereisten ID (VoorwaardeID) is uniek
df_vereisten.drop_duplicates(subset=['vraag_id'], inplace=True)

# Type vraag/antwoord is combinatie van kolom Schaal en AnalyseVariabele
# Coalesce kolom Schaal (prioritair) en AnalyseVariabele (fallback)
df_vereisten['schaal_type'] = df_vereisten.apply(lambda row: row['Schaal'] if pd.notnull(row['Schaal']) else row['AnalyseVariabele'], axis=1)

if len(df_vereisten[df_vereisten['schaal_type'].isnull()]) > 0:
    print("Er zijn vereisten zonder type vraag/antwoord. Controleer de invoervereisten!")
    print(df_vereisten[df_vereisten['schaal_type'].isnull()])

# We filter on specific habitat types (start with 3,4 or 5) to keep survey sizeable
# df_vereisten = df_vereisten[df_vereisten['Habitatsubtype'].astype(str).str.startswith(('1', '2','3', '4', '5'))]
df_vereisten = df_vereisten[df_vereisten['Habitatsubtype'].astype(str).str.startswith(tuple(['5','6','7']))]

# Haal beschrijving op uit LSVI databank (mimic functie geefInfoHabitatfiche)
df_vereisten = utils.voeg_lsvi_beschrijving_toe(df_vereisten, sqlite_path=SQLITE_PATH)

print(df_vereisten.shape)

print("Aantal unieke schaal in de vereisten:", df_vereisten['schaal_type'].unique())

df_soorten = pd.read_csv('../input/invoervereistenUitTeWerkenSoortenlijst-UTF8.csv', sep=';')
df_schalen = pd.read_csv('../input/survey123_schalen.csv', sep=';') # Het bestand van de vorige stap!

(710, 39)
(261, 42)
Aantal unieke schaal in de vereisten: <ArrowStringArray>
['LSVI', 'meting', 'meting_bedekking', 'meting_m']
Length: 4, dtype: str


In [149]:
# Groepen voor matrixvragen
df_groepen = pd.read_excel('../input/LSVI_packageInvoervereisten_uitdb_2026-06-08_aanvullingenLSVI_app.xlsx', sheet_name='Groepen')
# 2. Opschonen: Verwijder eventuele onzichtbare spaties aan de randen van de tekst
df_groepen['Name'] = df_groepen['Name'].astype(str).str.strip()
df_groepen['Value'] = df_groepen['Value'].astype(str).str.strip()
groepen_mapping = df_groepen.groupby('Name')['Value'].apply(list).to_dict()

# Habitattypes
df_habitattypes = pd.read_sql_table('Habitattype', 'sqlite:///../input/LSVIHabitatTypes.sqlite')

# BWK karteringseenheden
# Not used anymore, but kept for reference
# df_bwk_kart_eenh = pd.read_csv('../input/bwk_karteringseenheden.csv', sep=';')

## 1. Choices

In [150]:
print("Choices tabblad opbouwen...")
choices_list = []

# A. Schalen toevoegen
for _, row in df_schalen.iterrows():
    choices_list.append(row.to_dict())

# B. Dynamische Habitattype lijst genereren (Voor de eerste vraag)
unieke_habitats = df_vereisten['Habitattype'].dropna().unique()
for hab in unieke_habitats:
    choices_list.append({
        "list_name": "lijst_habitats",
        "name": utils.clean_name(hab),
        "label": str(hab).upper()
    })

# C. Dynamische Soortenlijsten genereren (Groepeer per TaxongroepId)
for _, soort in df_soorten.iterrows():
    if pd.notna(soort['TaxongroepId']):
        tax_id = int(soort['TaxongroepId'])
        choices_list.append({
            "list_name": f"taxa_{tax_id}",
            "name": utils.clean_name(soort['WetNaamKort']),
            "label": f"{soort['NedNaam']} ({soort['WetNaamKort']})" if pd.notna(soort['NedNaam']) else soort['WetNaamKort']
        })

# Dynamische subhapitattypes toevoegen (Voor de BWK-vragen)
unieke_subhabitats = df_vereisten['Habitatsubtype'].dropna().unique()
for hab in unieke_subhabitats:
    choices_list.append({
        "list_name": "lijst_subhabitats",
        "name": utils.clean_name(hab),
        "label": str(hab).upper()
    })

# BWK Karteringseenheden toevoegen
# Not used anymore
# for _, row in df_bwk_kart_eenh.iterrows():
#     choices_list.append({
#         "list_name": "lijst_bwk_karteringseenheden",
#         "name": utils.clean_name(row["karteringseenheid"]),
#         "label": str(row['karteringseenheid'])
#     })

# BWK Verhoudingen toevoegen
# Not used anymore
# bwk_verhoudingen = ['1 op 2', '2 op 3', '3 op 4', '4 op 5']
# for verhouding in bwk_verhoudingen:
#     choices_list.append({
#         "list_name": "lijst_bwk_verhoudingen",
#         "name": utils.clean_name(verhouding),
#         "label": verhouding
#     })
    
df_choices_final = pd.DataFrame(choices_list)

# Zorg dat de talen overeenkomen in survey en choices!
if 'label' in df_choices_final.columns:
    df_choices_final.rename(columns={'label': 'label::nl'}, inplace=True)



Choices tabblad opbouwen...


### Settings

In [151]:
# Add settings dataframe
df_settings = pd.DataFrame([{
    "form_id": "lsvi_app_test",
    "form_title": "LSVI App Test",
    "style": "pages",   # <-- DIT ACTIVEERT HET GRID SYSTEEM IN DE APP
    "default_language": "Dutch (nl)" 
}])

## 2. Survey

### 2.1 Algemene info

In [152]:
print("Survey tabblad opbouwen...")
survey_list = []

# Unieke ID: Wordt op de achtergrond gegenereerd (gebruiker ziet dit niet)
survey_list.append({
    "type": "calculate", "name": "collectie_id", "label": "", 
    "calculation": "uuid()", "relevant": "", "appearance": "", "default": ""
})

# Datum & Uur: Automatisch ingevuld
survey_list.append({
    "type": "date", "name": "datum", "label": "Datum", 
    "default": "today()", "relevant": "", "appearance": "", "calculation": ""
})

survey_list.append({
    "type": "time", "name": "uur", "label": "Uur", 
    "default": "now()", "relevant": "", "appearance": "", "calculation": ""
})

# Locatie: Onzichtbare GPS-bepaling
survey_list.append({
    "type": "geopoint", "name": "locatie", "label": "Locatie", 
    "appearance": "hidden", "default": "", "relevant": "", "calculation": ""
})

Survey tabblad opbouwen...


### 2.2 BWK-vragen

**In finale versie kunnen velden op appearance: hidden gezet worden.**

In [153]:
# Toon BWK ID over hele breedte
survey_list.append({
    "type": "integer", 
    "name": "bwk_id", 
    "label": "BWK ID", 
    "default": "", 
    "relevant": "", 
    "appearance": "",  # Neemt 5/5 kolommen in, dus over hele rij
    "calculation": "", 
    "readonly": "yes"
})

# Maak grid aan voor hab and phab velden weer te geven
survey_list.append({
    "type": "begin group", 
    "name": "grp_fieldmaps_data", 
    "label": "Gegevens uit Field Maps (Controle)", 
    "relevant": "", 
    "appearance": "w2 fixed-grid"  # <-- Dit activeert het grid-systeem!
})

survey_list.append({
    "type": "note", 
    "name": "hdr_hab", 
    "label": "<b>Habitat Code</b>", 
    "relevant": "", 
    "appearance": "w1", 
    "calculation": ""
})
survey_list.append({
    "type": "note", 
    "name": "hdr_phab", 
    "label": "<b>Percentage (%)</b>", 
    "relevant": "", 
    "appearance": "w1", 
    "calculation": ""
})


# Toon HAB en PHAB in grid
for i in range(1, 6):
    # Top row: Habitat Code (Breedte = w1)
    survey_list.append({
        "type": "text", 
        "name": f"hab{i}", 
        "label": "&nbsp;", #f"Hab. {i}", 
        "default": "", 
        "relevant": "", 
        "appearance": "w1",  # <-- Neem 1/2 kolommen in
        "calculation": "", 
        "readonly": "no" # Joost set to yes later on
    })

    # Bottom row: Percentage (Breedte = w1)
    survey_list.append({
        "type": "integer", 
        "name": f"phab{i}", 
        "label": "&nbsp;", #f"% Hab. {i}", 
        "default": "", 
        "relevant": "", 
        "appearance": "w1",  # <-- Neemt de andere helft van de ruimte in
        "calculation": "", 
        "readonly": "yes"
    })

# 4. Sluit de grid groep netjes af
survey_list.append({
    "type": "end group", "name": "", "label": "", "relevant": "", "appearance": ""
})

### 2.3 Vragen per habitattype

In [154]:
# Trigger vraag
survey_list.append({
    "type": "select_one Ja_Nee", "name": "lsvi_opstellen", "label": "LSVI Opstellen?", 
    "relevant": "", "appearance": "horizontal", "default": "", "calculation": ""
})

# Groepeer alle LSVI vragen zodat we de 'relevant' logica maar 1 keer hoeven te typen
survey_list.append({
    "type": "begin group", "name": "grp_lsvi", "label": "LSVI Gegevens", 
    "relevant": "${lsvi_opstellen} = 'ja'", # Zichtbaar als vorige vraag 'ja' is
    "appearance": "field-list", "default": "", "calculation": ""
})

# Eerste hoofdvraag: Welk habitattype?
survey_list.append({
    "type": "select_multiple lijst_subhabitats",
    "name": "habitat_keuze",
    "label": "Welk habitat(sub)type wil je inventariseren?",
    "relevant": "",  # Altijd zichtbaar
    # "hint": "Eenvoudige tekst om feature te testen.",
    # "guidance_hint": "Eenvoudige tekst om feature te testen.",
    "appearance": "horizontal", #blank defaults to radio buttons instead of "minimal autocomplete",
    "choice_filter": "string(name) = string(${hab1}) or string(name) = string(${hab2}) or string(name) = string(${hab3}) or string(name) = string(${hab4}) or string(name) = string(${hab5})" # This makes sure we only get to choose habitats that were mapped in BWK field app for this polygon.
})



# 4.2. Loop door de unieke habitattypes (Creëer "Pages" / Groups)
for hab in unieke_subhabitats:
    hab_clean = utils.clean_name(hab)

    # Eerste repeat blok aanmaken zodat vragen in apparte tabel worden opgeslagen
    # Feature layer is beperkt tot 1024 kolommen (dus vragen)
    # Maken aparte tabel aan per habitattype om deze limiet te omzeilen
    # survey_list.append({
    #     "type": "begin repeat",
    #     "name": f"rpt_habitat_{hab_clean}",  # 'rpt_' prefix helpt om database-tabellen te herkennen
    #     "label": "", # Keep label empty so it does not show in app
    #     "relevant": f"selected(string(${{habitat_keuze}}), '{hab_clean}')",   # De groep erft de relevantie van het repeat blok. Dit mag leeg zijn als we repeats gebruiken.
    #     "appearance": "minimal",  # Hakt de '+' en '-' knoppen weg voor de gebruiker
    #     "repeat_count": "1"        # Dwingt exact 1 gerelateerd record af (geen duplicaten mogelijk)
    # })

    # Vragen per habitattype
    # Begin de groep voor dit specifieke habitattype. 
    # De "relevant" zorgt ervoor dat enkel de habitattypes uit BWK-kaart worden weergegeven in app.
    survey_list.append({
        "type": "begin group",
        "name": f"grp_habitat_{hab_clean}",
        "label": f"Habitat {hab_clean.upper()}",
        # "hint": utils.get_habitat_hint(hab, db_path=SQLITE_PATH),
        # "guidance_hint": get_habitat_hint(hab),   # Dynamische hint genereren op basis van de habitattype code
        "relevant": f"selected(string(${{habitat_keuze}}), '{hab_clean}')",   # De groep erft de relevantie van het repeat blok. Dit mag leeg zijn als we repeats gebruiken.
        "appearance": "w1 compact field-list" # Zorgt dat het als 1 pagina toont in de app
    })

    # Filter de vereisten voor dít specifieke habitattype
    df_hab_vereisten = df_vereisten[df_vereisten['Habitatsubtype'] == hab]
    
    # 4.3. Genereer de vragen binnen dit habitattype
    for idx, row in df_hab_vereisten.iterrows():
        vraag_naam = f"{row['vraag_id']}"

        # Type_vraag heeft 3 mogelijkheden: Orig (normaal), Matrixvraag of 'niet nodig in app':
        if row['Type_vraag'].lower() == 'orig':
            # Do usual processing
            answer_type, vraag_appearance = utils.get_question_settings(row)

            # Controleer of er een beschrijving is
            heeft_beschrijving = pd.notna(row['Beschrijving']) and str(row['Beschrijving']).strip() != ""

            # Vraag toevoegen
            # Label van de vraag is combinatie van Voorwaarde + Indicator + Beoordeling(en eventueel Eenheid)
            # Add soortenlijst als hint 
            survey_list.append({
                "type": answer_type,
                "name": vraag_naam,
                "label": utils.get_question_label(row), # De vraag die de gebruiker ziet
                "relevant": "",
                "hint": np.nan,
                "appearance": vraag_appearance
            })
        
            # 2. Als er een LSVI-beschrijving is, maken we een schone, uitklapbare balk direct onder de vraag
            if pd.notna(row['Beschrijving']) and str(row['Beschrijving']).strip() != "":
                
                # Start de inklapbare groep
                survey_list.append({
                    "type": "begin group",
                    "name": f"besch_{vraag_naam}",
                    "label": "ℹ️ Bekijk indicator beschrijving", # De tekst op de klikbare balk
                    "relevant": "",
                    "appearance": "compact"  # <--- Dit zorgt ervoor dat hij standaard ingeklapt is!
                })

                # Voeg de note toe met de VOLLEDIGE beschrijving
                survey_list.append({
                    "type": "note",
                    "name": f"note_besch_{vraag_naam}",
                    "label": str(row['Beschrijving']).strip(), # De volledige, onafgekapte tekst
                    "relevant": "",
                    "appearance": "",
                    "bind::esri:fieldType": "null" # Zorgt ervoor dat dit geen lege GIS-kolom wordt
                })

                # Sluit de groep netjes af
                survey_list.append({
                    "type": "end group", "name": f"besch_{vraag_naam}", "label": np.nan, "relevant": "", "appearance": ""
                })

        # # Block below
        # elif row['Type_vraag'].lower() == 'matrixvraag':

        #     # Controleer of er een beschrijving is
        #     heeft_beschrijving = pd.notna(row['Beschrijving']) and str(row['Beschrijving']).strip() != ""
    
        #     # 1. Start subgroep voor matrix met de table-list layout
        #     survey_list.append({
        #         "type": "begin group",
        #         "name": f"{vraag_naam}_matrix",
        #         "label": utils.get_question_label(row), # Genereert jouw mooie HTML label
        #         "hint": np.nan,
        #         "relevant": "",
        #         "appearance": "table-list" # <-- GEWIJZIGD: Verander 'w2 grid-layout' naar 'table-list'
        #     })

        #     # Welke groep moeten we bevragen in matrix?
        #     groep_naam = str(row['Groepen']).strip().lower()
        #     items_te_scoren = []
            
        #     if 'sleutelsoorten' in groep_naam:
        #         tax_id = row['TaxongroepId']
        #         if pd.notna(tax_id):
        #             df_sub_soorten = df_soorten[df_soorten['TaxongroepId'] == int(tax_id)]
        #             items_te_scoren = df_sub_soorten['NedNaam'].fillna(df_sub_soorten['WetNaam']).tolist()
        #     else:
        #         raw_groep = str(row['Groepen']).strip()
        #         items_te_scoren = groepen_mapping.get(raw_groep, [])

        #     if not items_te_scoren:
        #         print(f"Waarschuwing: Geen matrix items gevonden voor groep '{row['Groepen']}' bij vraag {vraag_naam}")

        #     # 2. Genereer de matrix rijen
        #     # Binnen een 'table-list' groep hoef je GEEN 'notes' toe te voegen voor de labels!
        #     for index, item in enumerate(items_te_scoren):
        #         uniek_veld_name = f"{vraag_naam}_matrix_{index}"
        #         uniek_veld_name = uniek_veld_name[0:27] # Beperkt tot 32 tekens max voor GIS/Excel kolommen

        #         # GEWIJZIGD: Geen aparte 'note' meer toevoegen. 
        #         # We voegen direct de 'select_one' vraag toe. Het label van deze vraag 
        #         # wordt door Survey123 automatisch als de linker rijkop geplaatst.
        #         survey_list.append({
        #             "type": "select_one LSVI", # Zorg dat al deze vragen exact dezelfde keuzelijst delen!
        #             "name": uniek_veld_name,
        #             "label": f"{item.capitalize()}", # <-- GEWIJZIGD: De itemnaam is nu direct het label van de keuzevraag
        #             "relevant": "",
        #             "appearance": "" # <-- GEWIJZIGD: Verwijder 'minimal' en 'w1'. table-list regelt de styling.
        #         })

        #     # 3. Sluit de matrix sub-groep netjes af
        #     survey_list.append({
        #         "type": "end group", "name": "", "label": "", "relevant": "", "appearance": ""
        #     })
        
        # elif row['Type_vraag'].lower() == 'matrixvraag':
        #     # Moet iets doen voor alle groepen? Zie groep kolom? 
            
        #     # Start subgroep voor matrix met grid layout
        #     survey_list.append({
        #         "type": "begin group",
        #         "name": f"{vraag_naam}_matrix",
        #         "label": utils.get_question_label(row), # Genereert jouw mooie HTML label
        #         "hint": "Scoor elk van de onderstaande onderdelen volgens de LSVI-schaal.",
        #         "relevant": "",
        #         "appearance": "w2 grid-layout" # Activeert het 2-koloms rastersysteem
        #     })

        #     # Welke groep moeten we bevragen in matrix? Sleutelsoorten of andere categorie? 
        #     groep_naam = str(row['Groepen']).strip().lower()
        #     items_te_scoren = []
            
        #     if 'sleutelsoorten' in groep_naam:
        #         # CASE A: Haal de specifieke soorten op uit df_soorten op basis van TaxongroepId
        #         tax_id = row['TaxongroepId']
        #         if pd.notna(tax_id):
        #             df_sub_soorten = df_soorten[df_soorten['TaxongroepId'] == int(tax_id)]
        #             # Pak 'NedNaam', tenzij NaN, dan pak 'WetNaam'
        #             items_te_scoren = df_sub_soorten['NedNaam'].fillna(df_sub_soorten['WetNaam']).tolist()

        #     else:
        #         # CASE B: Haal de vaste categorieën op uit het tabblad 'Groepen' (de dictionary)
        #         raw_groep = str(row['Groepen']).strip()
        #         items_te_scoren = groepen_mapping.get(raw_groep, [])

        #     # Veiligheidscheck voor als er niets gevonden is
        #     if not items_te_scoren:
        #         print(f"Waarschuwing: Geen matrix items gevonden voor groep '{row['Groepen']}' bij vraag {vraag_naam}")

        #     # Genereer de rijen (Text + Dropdown paren) binnen het grid
        #     for index, item in enumerate(items_te_scoren):
        #         # We saneren de tekst handmatig naar kleine letters zonder vreemde tekens
        #         item_clean = re.sub(r'[^a-z0-9]', '', str(item).lower().strip())
        #         voorwaarde_clean = re.sub(r'[^a-z0-9]', '', str(row['VoorwaardeID']).lower().strip())
                
        #         # SLIMME TRUC: Omdat jouw utils.clean_name strikt maximaal 2 delen (Deel1_Deel2) 
        #         # toestaat, bouwen we de veldnaam op als "v{VoorwaardeID}_{item}". 
        #         # Dit levert exact 2 delen op (bijv: v317_pionierstadium) wat perfect uniek is!
        #         uniek_veld_name = f"{vraag_naam}_matrix_{index}"
        #         uniek_veld_name = uniek_veld_name[0:27] # Beperkt tot 32 tekens

        #         # KOLOM 1 (Links): De naam van de soort of het stadium (w6 = 50% breedte)
        #         survey_list.append({
        #             "type": "note",
        #             "name": f"{uniek_veld_name}_note", # Beperkt tot 32 tekens
        #             "label": f"{item.capitalize()}",
        #             "relevant": "",
        #             "appearance": "w1" 
        #         })
                
        #         # KOLOM 2 (Rechts): De LSVI Keuzelijst Dropdown (w6 = 50% breedte)
        #         survey_list.append({
        #             "type": "select_one LSVI",
        #             "name": uniek_veld_name,
        #             "label": "Score:", # Spatie verbergt het label visueel, maar voorkomt Connect fouten
        #             "relevant": "",
        #             "appearance": "minimal w1" # 'minimal' maakt er een dropdown van
        #         })

        #     # 4. Sluit de matrix sub-groep netjes af
        #     survey_list.append({
        #         "type": "end group", "name": "", "label": "", "relevant": "", "appearance": ""
        #     })

        elif row['Type_vraag'].lower() == 'matrixvraag':
            vraag_naam = f"{row['vraag_id']}"
            
            # 1. Start de matrix hoofdgroep met 'field-list' (verticale stapeling over 100% breedte)
            survey_list.append({
                "type": "begin group",
                "name": f"{vraag_naam}_matrix",
                "label": utils.get_question_label(row), # Jouw mooie HTML hoofdlabel
                "hint": "Scoor elk van de onderstaande onderdelen volgens de LSVI-schaal.",
                "relevant": "",
                "appearance": "field-list" # <-- GEWIJZIGD: field-list stapt af van het krappe grid
            })

            # 2. VOEG DE INKLAPBARE INDICATOR-BESCHRIJVING TOE (Bovenin de groep)
            if pd.notna(row['Beschrijving']) and str(row['Beschrijving']).strip() != "":
                # Start de inklapbare subgroep
                survey_list.append({
                    "type": "begin group",
                    "name": f"grp_besch_{vraag_naam}",
                    "label": "ℹ️ Bekijk indicator beschrijving", 
                    "relevant": "",
                    "appearance": "compact"  # Standaard netjes ingeklapt
                })

                # De note met de volledige tekst
                survey_list.append({
                    "type": "note",
                    "name": f"note_besch_{vraag_naam}",
                    "label": str(row['Beschrijving']).strip(),
                    "relevant": "",
                    "appearance": "",
                    "bind::esri:fieldType": "null"
                })

                # Sluit de inklapbare subgroep
                survey_list.append({
                    "type": "end group", "name": f"grp_besch_{vraag_naam}", "label": "", "relevant": "", "appearance": ""
                })

            # 3. Welke groep moeten we bevragen? (Sleutelsoorten of vaste mapping)
            groep_naam = str(row['Groepen']).strip().lower()
            items_te_scoren = []
            
            if 'sleutelsoorten' in groep_naam:
                tax_id = row['TaxongroepId']
                if pd.notna(tax_id):
                    df_sub_soorten = df_soorten[df_soorten['TaxongroepId'] == int(tax_id)]
                    items_te_scoren = df_sub_soorten['NedNaam'].fillna(df_sub_soorten['WetNaam']).tolist()
            else:
                raw_groep = str(row['Groepen']).strip()
                items_te_scoren = groepen_mapping.get(raw_groep, [])

            if not items_te_scoren:
                print(f"Waarschuwing: Geen matrix items gevonden voor groep '{row['Groepen']}' bij vraag {vraag_naam}")

            # 4. Genereer de rijen over de VOLLEDIGE breedte van het scherm
            for index, item in enumerate(items_te_scoren):
                uniek_veld_name = f"{vraag_naam}_matrix_{index}"
                uniek_veld_name = uniek_veld_name[0:27] # Beperkt tot 32 tekens max

                # OPLOSSING VOOR SMALLE SCHERMEN:
                # - We verwijderen de aparte 'note' rij volledig.
                # - De soortnaam/stadium-naam wordt DIRECT het 'label' van de select_one vraag.
                # - We halen 'w1' weg. 'appearance: minimal' zorgt nu voor een dropdown
                #   die de volle 100% breedte van het scherm gebruikt.
                survey_list.append({
                    "type": "select_one LSVI",
                    "name": uniek_veld_name,
                    "label": f"{item.capitalize()}", # De soortnaam staat nu groot en leesbaar BOVEN de dropdown
                    "relevant": "",
                    "appearance": "minimal" # Volledige breedte dropdown, perfect voor mobiel
                })

            # 5. Sluit de matrix hoofdgroep netjes af
            survey_list.append({
                "type": "end group", "name": "", "label": "", "relevant": "", "appearance": ""
            })


        else:
            # Skip question
            continue
        
    
        # Onder vraag wil je soms de soortenlijst weergeven.
        # Doen dit niet als hint of guidance_hint in de vraag, want niet inklapbaar of te weinig ruimte.
        # We maken een aparte groep aan (conditioneel) met soorten indien nodig.
        # Genereer de soortenlijst HTML
        if row["Soortenlijst weergeven in vraag"] == 1:
            html_soorten = utils.get_species_hint(row['TaxongroepId'], df_soorten)

            # Check of er een soortenlijst is om te tonen
            if pd.notna(html_soorten) and str(html_soorten).strip() != "":
                # Start de in- en uitklapbare groep
                survey_list.append({
                    "type": "begin group",
                    "name": f"grp_lijst_{vraag_naam}",
                    "label": "🔽 Bekijk soortenlijst", # Dit is de tekst op de klikbare balk
                    "hint": np.nan,
                    # "guidance_hint": np.nan,
                    "relevant": "",
                    "appearance": "compact" # <--- Dit commando maakt hem standaard ingeklapt!
                })

                # Voeg de 'note' toe met jouw HTML-geformatteerde soortenlijst
                survey_list.append({
                    "type": "note",
                    "name": f"note_{vraag_naam}",
                    "label": html_soorten, # We stoppen jouw HTML nu in de 'label' van de note
                    "hint": np.nan,
                    # "guidance_hint": np.nan,
                    "relevant": "",
                    "appearance": "",
                    "bind::esri:fieldType": "null"
                })

                # Sluit de uitklapbare groep netjes af
                survey_list.append({
                    "type": "end group",
                    "name": "", 
                    "label": np.nan, 
                    "hint": np.nan, 
                    # "guidance_hint": np.nan,
                    "relevant": "", 
                    "appearance": ""
                })

    # Sluit de groep (Pagina) af
    survey_list.append({
        "type": "end group",
        "name": "", "label": "", "relevant": "", "appearance": ""
    })

    # # Sluit repeat block af
    # survey_list.append({
    #     "type": "end repeat", "name": ""
    # })

# Sluit de LSVI Groep af
survey_list.append({
    "type": "end group", "name": "", "label": "", 
    "relevant": "", "appearance": "", "default": "", "calculation": ""
})

df_survey_final = pd.DataFrame(survey_list)

In [155]:
# df_survey_final[
#     ~((df_survey_final['name'].str.startswith('grp_', na=False))
#       | (df_survey_final['name'].str.startswith('rpt_', na=False))
#       | (df_survey_final['name'].str.startswith('note_', na=False))
#       | (df_survey_final['name'].isna())
#       | (df_survey_final['name'] == '')
#       )
#     ].head(30)

df_fl_clean = df_survey_final[
    ~((df_survey_final['name'].str.startswith('grp_', na=False))
      | (df_survey_final['name'].str.startswith('rpt_', na=False))
      | (df_survey_final['name'].str.startswith('note_', na=False))
      | (df_survey_final['name'].isna())
      | (df_survey_final['name'] == '')
      )].reset_index(drop=True)

df_fl_clean['voorwaarde_id'] = df_fl_clean['name'].str.extract(r'vrg_(\d+)_')
df_fl_clean.iloc[20:40]

# # start of habitat 6  voorwaarde 1166
# # index where voorwaarde_id == 1166
# habitat_6_index_start = df_fl_clean[df_fl_clean['voorwaarde_id'] == '1166'].index[0]
# print(f"Index of habitat 6 voorwaarde 1166: {habitat_6_index_start}")
# # end of habitat 6 voorwaarde 1304
# habitat_6_index_end = df_fl_clean[df_fl_clean['voorwaarde_id'] == '1304'].index[0]
# print(f"Index of habitat 6 voorwaarde 1304: {habitat_6_index_end}")

# # Habitat 6 probably to large for 1 survey

# # start of habitat 9  voorwaarde 1949
# # index where voorwaarde_id == 1949
# habitat_9_index_start = df_fl_clean[df_fl_clean['voorwaarde_id'] == '1949'].index[0]
# print(f"Index of habitat 9 voorwaarde 1949: {habitat_9_index_start}")
# # end of habitat 9 voorwaarde 538
# habitat_9_index_end = df_fl_clean[df_fl_clean['voorwaarde_id'] == '538'].index[0]
# print(f"Index of habitat 9 voorwaarde 538: {habitat_9_index_end}")

,type,name,label,calculation,relevant,appearance,default,readonly,choice_filter,hint,bind::esri:fieldType,voorwaarde_id
20,begin group,besch_vrg_732_5130_hei,ℹ️ Bekijk indicator beschrijving,NaN,,compact,NaN,NaN,NaN,NaN,NaN,732
21,end group,besch_vrg_732_5130_hei,NaN,NaN,,,NaN,NaN,NaN,NaN,NaN,732
22,decimal,vrg_150_5130_hei,<b>Indicator:</b> <span style='font-weight: no...,NaN,,minimal,NaN,NaN,NaN,NaN,NaN,150
23,begin group,besch_vrg_150_5130_hei,ℹ️ Bekijk indicator beschrijving,NaN,,compact,NaN,NaN,NaN,NaN,NaN,150
24,end group,besch_vrg_150_5130_hei,NaN,NaN,,,NaN,NaN,NaN,NaN,NaN,150
25,decimal,vrg_218_5130_hei,<b>Indicator:</b> <span style='font-weight: no...,NaN,,minimal,NaN,NaN,NaN,NaN,NaN,218
26,begin group,besch_vrg_218_5130_hei,ℹ️ Bekijk indicator beschrijving,NaN,,compact,NaN,NaN,NaN,NaN,NaN,218
27,end group,besch_vrg_218_5130_hei,NaN,NaN,,,NaN,NaN,NaN,NaN,NaN,218
28,select_one LSVI,vrg_1591_5130_hei,<b>Indicator:</b> <span style='font-weight: no...,NaN,,compact horizontal,NaN,NaN,NaN,NaN,NaN,1591
29,select_one LSVI,vrg_1741_5130_hei,<b>Indicator:</b> <span style='font-weight: no...,NaN,,compact horizontal,NaN,NaN,NaN,NaN,NaN,1741


### 2.4 Export to XLSX

In [156]:

# Vervang alle lege strings in de hint (en eventueel andere) kolommen door echte NaN waarden
df_survey_final['hint'] = df_survey_final['hint'].replace("", np.nan)
# df_survey_final['guidance_hint'] = df_survey_final['guidance_hint'].replace("", np.nan)

# rename columns to add default language
# df_survey_final = df_survey_final.rename(columns={'label': 'label::nl', 'hint': 'hint::nl', 'guidance_hint': 'guidance_hint::nl'})

print("Excel bestand genereren...")
output_file = '../output/Survey123_Volledige_Generatie_20260713.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_survey_final.to_excel(writer, sheet_name='survey', index=False)
    df_choices_final.to_excel(writer, sheet_name='choices', index=False)
    # Voeg een standaard settings tabblad toe
    df_settings.to_excel(writer, sheet_name='settings', index=False)

# Overwrite Survey123 test app
output_file = r'C:\Users\joost_neujens\ArcGIS\My Survey Designs\LSVI App Test\LSVI App Test.xlsx' # if running local
# output_file = r"\\Client\C$\Users\joost_neujens\ArcGIS\My Survey Designs\LSVI App Test\LSVI App Test.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_survey_final.to_excel(writer, sheet_name='survey', index=False)
    df_choices_final.to_excel(writer, sheet_name='choices', index=False)
    # Voeg een standaard settings tabblad toe
    df_settings.to_excel(writer, sheet_name='settings', index=False)

print(f"✅ Survey123 XLSForm succesvol gegenereerd: '{output_file}'")

Excel bestand genereren...
✅ Survey123 XLSForm succesvol gegenereerd: 'C:\Users\joost_neujens\ArcGIS\My Survey Designs\LSVI App Test\LSVI App Test.xlsx'


In [127]:
df_survey_final.shape

(1502, 12)

In [144]:
df_vereisten[(df_vereisten['Type_vraag'] == 'Matrixvraag')][['Habitattype','Habitatsubtype','BeoordelingID','TaxongroepId','Beschrijving']] # & (df_vereisten['Beschrijving'].notna())]

,Habitattype,Habitatsubtype,BeoordelingID,TaxongroepId,Beschrijving
21,6120,6120,1539,350,NaN
29,6210,6210,1507,351,NaN
38,6230,6230_ha,1563,353,NaN
49,6230,6230_hmo,1559,352,NaN
58,6230,6230_hn,1582,353,NaN
67,6230,6230_hnk,1599,931,NaN
68,6230,6230_hnk,1599,193,NaN
69,6230,6230_hnk,1599,241,NaN
82,6410,6410,2249,937,NaN
90,6430,6430_bz,1618,355,NaN


In [141]:
df_vereisten[df_vereisten['BeoordelingID'] == 1539]

,Unnamed: 0,Versie,Habitattype,Habitatsubtype,Criterium,Indicator,Beoordeling,Kwaliteitsniveau,Belang,BeoordelingID,Combinatie,VoorwaardeID,Voorwaarde,Referentiewaarde,Opm (te checken),Type_vraag,Groepen,Soortenlijst weergeven in vraag,Soortenlijst volledig?,Schaal,Operator,Maximumwaarde,AnalyseVariabele,Eenheid,TypeVariabele,Invoertype,Invoerwaarde,TaxongroepId,TaxongroepNaam,Studiegroepnaam,Studielijstnaam,Studiewaarde,SubAnalyseVariabele,SubEenheid,TypeSubVariabele,SubReferentiewaarde,SubOperator,SubInvoertype,SubInvoerwaarde,vraag_id,schaal_type,Beschrijving
21,602,Versie 3,6120,6120,Vegetatie,sleutelsoorten,>= 4,1,b,1539,431,431,aantal sleutelsoorten,4,NaN,Matrixvraag,sleutelsoorten,NaN,NaN,LSVI,>=,12,aantal,NaN,Geheel getal,NaN,NaN,350,sleutelsoorten,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,vrg_431_6120,LSVI,NaN


In [ ]:
df_survey_final[df_survey_final['']]

,type,name,label,calculation,relevant,appearance,default,readonly,choice_filter,guidance_hint,hint,bind::esri:fieldType
50,begin group,grp_habitat_6120,Habitat 6120,NaN,"selected(string(${habitat_keuze}), '6120')",w1 compact field-list,NaN,NaN,NaN,NaN,NaN,NaN
51,select_one LSVI,vrg_1166_6120,<b>Indicator:</b> <span style='font-weight: no...,NaN,,compact horizontal,NaN,NaN,NaN,"aandeel vegetatieloze bodem, exclusief eventue...",NaN,NaN
52,select_one LSVI,vrg_1159_6120,<b>Indicator:</b> <span style='font-weight: no...,NaN,,compact horizontal,NaN,NaN,NaN,"aandeel vegetatieloze bodem, exclusief eventue...",NaN,NaN
53,select_one LSVI,vrg_863_6120,<b>Indicator:</b> <span style='font-weight: no...,NaN,,compact horizontal,NaN,NaN,NaN,NaN,NaN,NaN
54,begin group,grp_lijst_vrg_863_6120,🔽 Bekijk soortenlijst,NaN,,compact,NaN,NaN,NaN,NaN,NaN,NaN
